In [1]:
# Imports

import ray
from ray.rllib.algorithms.ppo import PPOConfig

import IPython.core.display_functions

from src.parsers import HMParser, CotevParser
from src.algorithms.rl import EnergyCommunityBaselineV3

from src.utils import load_multiple_upacs_pv, iterate_resources, create_ppo_policies

import warnings
warnings.filterwarnings('ignore')

2025-02-02 20:06:41,747	WARNING deprecation.py:50 -- DeprecationWarning: `DirectStepOptimizer` has been deprecated. This will raise an error in the future!


In [2]:
# Data parsing

# EC data for non-renewable generators and batteries
data_ec = HMParser(file_path='/Users/ecgomes/DataspellProjects/pyecom/data/EC_V4.xlsx', ec_id=1)
data_ec.parse()

# EV data from the EV4EU simulator
data_ev = CotevParser(population_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/population_731.csv',
                      driving_history_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/ev_driving_history_731.csv',
                      assigned_segments_path='/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/assigned_segments_731.csv',
                      parse_date_start='2019',
                      parse_date_end='2020')
data_ev.parse()

# UPAC Data load
data_upacs = load_multiple_upacs_pv('/Users/ecgomes/Documents/PhD/UPAC data/upac*_pv.csv', resample='H')

In [3]:
# Create resources for the training environment

dataset_resources = iterate_resources(u=data_upacs, c=data_ec, e=data_ev, mode='monthly')

In [4]:
# Create the environment and check if everything is ok

temp_env = EnergyCommunityBaselineV3(ren_generators=dataset_resources[list(dataset_resources.keys())[0]][:5],
                                     generators=[],
                                     loads=dataset_resources[list(dataset_resources.keys())[0]][5:10],
                                     storages=dataset_resources[list(dataset_resources.keys())[0]][10:13],
                                     evs=dataset_resources[list(dataset_resources.keys())[0]][13:-1],
                                     aggregator=dataset_resources[list(dataset_resources.keys())[0]][-1],
                                     storage_penalty=1,
                                     ev_penalty=1,
                                     balance_penalty=1)
temp_env.reset()
terminations = truncations = {a: False for a in temp_env.agents}
terminations['__all__'] = False
truncations['__all__'] = False
while not terminations['__all__'] and not truncations['__all__']:

    actions = temp_env.action_space_sample()
    next_obs, rewards, terminations, truncations, infos = temp_env.step(actions)

print('Terminated: {}'.format(terminations['__all__']))

Terminated: True


In [5]:
# Create the policies to train

# The keys of the dictionary respect the class names of the agents
gammas = {'Generator': 0.0, 'Storage': 0.9, 'Vehicle': 0.9, 'Aggregator': 0.9}

# Create the policies, one for each agent. Each policy has the name of the agent.
policies = create_ppo_policies(temp_env, gammas)

2025-02-02 20:07:40,398	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-02-02 20:07:40,399	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-02-02 20:07:40,400	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module

In [6]:
# Create an RLlib Algorithm instance from a PPOConfig to learn how to
# act in the above environment.

from ray.tune import register_env
from ray import tune, train
from ray.tune.schedulers import AsyncHyperBandScheduler
from ray.tune.stopper import CombinedStopper, MaximumIterationStopper, TrialPlateauStopper

ray.shutdown()
ray.init()

IMPORT_PENALTY = 1 #100
EXPORT_PENALTY = 1 #10
STORAGE_ACTION_PENALTY = 1 #100
STORAGE_ACTION_REWARD = 5 #10
EV_ACTION_PENALTY = 1 #1000
EV_ACTION_REWARD = 5 #10
EV_REQUIREMENT_PENALTY = 2000 #3000
BALANCE_PENALTY = 5000 #20000

MAX_ITER = 500

checkpoint = None
checkpoint_path = None
algo = None
current_best = None

# Build a loop for using separate resources on a daily basis
for datapoint in list(dataset_resources.keys())[:1]:

    temp_resources = dataset_resources[datapoint]

    env = EnergyCommunityBaselineV3(ren_generators=temp_resources[:5],
                                    generators=[],
                                    loads=temp_resources[5:10],
                                    storages=temp_resources[10:13],
                                    evs=temp_resources[13:-1],
                                    aggregator=temp_resources[-1],
                                    storage_penalty=STORAGE_ACTION_PENALTY,
                                    ev_penalty=EV_REQUIREMENT_PENALTY,
                                    balance_penalty=BALANCE_PENALTY)
    register_env("EC_Baseline_V0", lambda config: env)

    # Define the PPOConfig
    config = PPOConfig() \
        .environment(env="EC_Baseline_V0", disable_env_checking=False) \
        .training(train_batch_size=512,
                  lr=2e-4, #tune.grid_search([0.001, 0.0001]),
                  gamma=0.99,
                  use_gae=True,
                  use_critic=True,
                  use_kl_loss=True,
                  clip_param=0.1,
                  model={'use_lstm': True,
                         'fcnet_hiddens': [256, 256],
                         'lstm_cell_size': 64,
                         #'max_seq_len': 20,
                         'lstm_use_prev_action': True,
                         'lstm_use_prev_reward': True,
                         'vf_share_layers': False
                         }
                  ) \
        .framework('torch') \
        .rollouts(batch_mode='complete_episodes', #'complete_episodes',
                  num_rollout_workers=10,
                  rollout_fragment_length='auto') \
        .multi_agent(policies=policies,
                     policy_mapping_fn=(lambda agent_id, episode, worker, **kwargs:
                                        agent_id))

    #algo = config.build()    

    # Clear the Jupyter cell output
    IPython.core.display_functions.clear_output()

    stopper = CombinedStopper(
        MaximumIterationStopper(max_iter=MAX_ITER),
        TrialPlateauStopper(metric='episode_reward_max', mode='max', metric_threshold=0.01)
    )

    scheduler = AsyncHyperBandScheduler(time_attr="training_iteration",
                                        max_t=MAX_ITER)

    # Train the algorithm with Tuner
    tuner = tune.Tuner(
        "PPO",
        param_space=config,
        run_config=train.RunConfig(stop={'training_iteration': MAX_ITER, 'episode_reward_mean': 4000}),
        tune_config=tune.TuneConfig(scheduler=scheduler,
                                    num_samples=1, metric="episode_reward_max", mode="max"),
    )

    results = tuner.fit()

(PPO pid=1098) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/Users/ecgomes/ray_results/PPO_2025-02-02_20-08-37/PPO_EC_Baseline_V0_7d40d_00000_0_2025-02-02_20-08-37/checkpoint_000000)
2025-02-03 03:16:15,639	INFO tune.py:1143 -- Total run time: 25657.73 seconds (25656.97 seconds for the tuning loop).


In [7]:
 results.get_best_result('episode_reward_mean', 'max').get_best_checkpoint('episode_reward_mean', 'max').path

'/Users/ecgomes/ray_results/PPO_2025-02-02_20-08-37/PPO_EC_Baseline_V0_7d40d_00000_0_2025-02-02_20-08-37/checkpoint_000000'